In [1]:
import pandas as pd
from lightgbm import LGBMClassifier
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import f1_score, make_scorer

df = pd.read_csv('../data/processed/fused.csv')

DROP = [
    c for c in
    ['target', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF']
    if c in df.columns
]

y = df['target']

IOT_COLS = [
    c for c in df.columns
    if c not in DROP
    and c not in [
        'ambient_temp_c',
        'factory_load_pct',
        'humidity_pct',
        'ambient_temp_c_roll_mean_5',
        'factory_load_pct_roll_mean_5',
        'humidity_pct_roll_mean_5'
    ]
]

ALL_COLS = [
    c for c in df.columns
    if c not in DROP
]

macro_f1 = make_scorer(
    f1_score,
    average='macro'
)

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)


def make_pipe():
    return ImbPipeline([
        (
            'smote',
            SMOTE(
                random_state=42,
                k_neighbors=3
            )
        ),
        (
            'clf',
            LGBMClassifier(
                n_estimators=500,
                learning_rate=0.05,
                class_weight='balanced',
                random_state=42,
                n_jobs=-1
            )
        )
    ])


s_iot = cross_val_score(
    make_pipe(),
    df[IOT_COLS],
    y,
    cv=skf,
    scoring=macro_f1,
    n_jobs=-1
).mean()

s_all = cross_val_score(
    make_pipe(),
    df[ALL_COLS],
    y,
    cv=skf,
    scoring=macro_f1,
    n_jobs=-1
).mean()

print(f"IoT only:       Macro F1 = {s_iot:.4f}")
print(f"IoT + External: Macro F1 = {s_all:.4f}")
print(f"Improvement:    +{(s_all - s_iot):.4f}")

IoT only:       Macro F1 = 0.9107
IoT + External: Macro F1 = 0.9032
Improvement:    +-0.0076


In [2]:
df = pd.read_csv('../data/processed/fused.csv')

corr = (
    df.corr(numeric_only=True)['target']
      .abs()
      .sort_values(ascending=False)
)

print(corr.head(15))

target                        1.000000
HDF                           0.575800
OSF                           0.531083
PWF                           0.522812
TWF                           0.362904
torque_nm                     0.191321
power_proxy                   0.176039
torque_nm_roll_max_5          0.151756
torque_nm_roll_std_5          0.132387
temp_diff                     0.111676
torque_nm_roll_max_10         0.110649
tool_wear_min_roll_mean_20    0.107908
tool_wear_min_roll_mean_10    0.107342
tool_wear_min_roll_mean_5     0.106058
tool_wear_min                 0.105448
Name: target, dtype: float64
